# Exercícios 01–06

**Prof. Daniel de Abreu Pereira Uhr**

**Curso:** Estatística e Econometria com Python


Daniel de Abreu Pereira Uhr · Capítulo 5

## Contexto — Cafeteria do exemplo

A cafeteria do exemplo, empresa fictícia, quer entender a relação entre a **temperatura máxima do dia**, em graus Celsius, e o **número de copos de bebidas geladas vendidos**. Ela registrou os dois valores em 90 dias. Os dados são didáticos.

Execute a célula abaixo antes de começar.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)
dias = pd.DataFrame({"temperatura_c": np.round(rng.uniform(18, 36, size=90), 1)})
dias["copos"] = np.round(40 + 6 * dias["temperatura_c"] + rng.normal(0, 25, size=90)).astype(int)

## Exercício 01 — O gráfico primeiro

Faça o gráfico de dispersão da temperatura (eixo horizontal) e dos copos vendidos (eixo vertical). Descreva direção, forma, intensidade e pontos atípicos.

## Exercício 02 — Covariância

Calcule a covariância à mão, a partir dos desvios em relação às médias, e confira com `Series.cov`. Conte quantos dias ficam em cada quadrante em torno das médias. Qual é a unidade da covariância?

## Exercício 03 — Correlação

Calcule a correlação de Pearson à mão e confira com `Series.corr`.

## Exercício 04 — Graus Fahrenheit

Converta a temperatura para graus Fahrenheit ($F = 1{,}8\,C + 32$) e recalcule a covariância e a correlação com os copos vendidos. O que mudou, o que não mudou e por quê?

## Exercício 05 — O que a correlação diz

Um sócio pergunta se a correlação obtida é “forte”. Responda sem usar faixas fixas: o que o número, o gráfico e o contexto permitem dizer? Cite algo que a correlação **não** permite concluir.

## Exercício 06 — O dia do festival

Em um dia de 21 °C, um festival na praça em frente à cafeteria levou a 520 copos vendidos. Acrescente esse dia aos dados e recalcule as correlações de Pearson e de Spearman (`method="spearman"`). Como esse dia deveria ser tratado e comunicado na análise?

# Exercícios 07–12

**Prof. Daniel de Abreu Pereira Uhr**

**Curso:** Estatística e Econometria com Python


Daniel de Abreu Pereira Uhr · Capítulo 5

## Contexto — Empresa de entregas

O empresa de entregas, aplicativo fictício de entregas, registrou em 120 dias: o tempo médio de entrega (minutos), a nota média dada pelos clientes (de 1 a 5), o número de pedidos, o número de entregadores escalados e o número de atrasos. Os dados são didáticos.

Execute a célula abaixo antes de começar. Use o mesmo gerador `rng` em todas as questões, na ordem em que aparecem.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)
n_dias = 120
dias = pd.DataFrame({"tempo_medio_min": np.round(rng.uniform(20, 60, size=n_dias), 1)})
dias["nota_media"] = np.round(4.7 - 0.0025 * np.maximum(dias["tempo_medio_min"] - 35, 0) ** 2 + rng.normal(0, 0.08, size=n_dias), 2)
dias["pedidos"] = np.round(rng.uniform(200, 1000, size=n_dias)).astype(int)
dias["entregadores"] = np.round(dias["pedidos"] / 25 + rng.normal(0, 4, size=n_dias)).astype(int)
dias["atrasos"] = np.round(np.maximum(0.08 * dias["pedidos"] - 1.5 * (dias["entregadores"] - dias["pedidos"] / 25) + rng.normal(0, 6, size=n_dias), 0)).astype(int)

## Exercício 07 — Tempo de entrega e nota

Faça o gráfico de dispersão do tempo médio de entrega e da nota média e calcule a correlação de Pearson. A relação parece linear?

## Exercício 08 — Dois regimes

Calcule a correlação separadamente para os dias com tempo médio abaixo de 35 minutos e a partir de 35 minutos. Calcule também a correlação de Spearman para todos os dias. O que a correlação de Pearson para todos os dias esconde?

## Exercício 09 — Mais entregadores, mais atrasos?

Calcule a correlação entre o número de entregadores escalados e o número de atrasos. Um gerente propõe: “quanto mais entregadores escalamos, mais atrasos temos; vamos reduzir a escala”. Que terceira variável pode explicar essa correlação?

## Exercício 10 — Comparar dias parecidos

Divida os dias em quatro faixas de número de pedidos (200–400, 400–600, 600–800 e 800–1.000) e calcule a correlação entre entregadores e atrasos dentro de cada faixa. Faça um gráfico de dispersão com uma cor ou tom por faixa. O que muda em relação a Exercício 09?

## Exercício 11 — Medidas relativas

Calcule os pedidos por entregador e os atrasos por 100 pedidos em cada dia e a correlação entre essas duas medidas. Interprete o resultado e compare com a proposta do gerente.

## Exercício 12 — O piloto de oito dias

Em um piloto de 8 dias, a equipe calculou correlação de 0,6 entre o bônus pago aos entregadores e a nota média, e concluiu que “o bônus melhora a avaliação”. Simule $R = 5.000$ amostras de 8 pares de variáveis **independentes** ($\rho = 0$) e calcule a frequência de $|r| \geq 0{,}6$. Use o resultado para avaliar a conclusão.

# Exercícios 13–18

**Prof. Daniel de Abreu Pereira Uhr**

**Curso:** Estatística e Econometria com Python


Daniel de Abreu Pereira Uhr · Capítulo 5

## Contexto — A lei de Engel

Esta aplicação usa **dados reais**: a renda anual e o gasto anual com alimentação, em francos belgas, de 235 famílias de trabalhadores na Bélgica, em 1857. A ficha `data/ch05/engel_1857_FONTE.md` registra fonte, condição de uso (domínio público), proveniência e limites.

Engel observou que famílias mais ricas gastam mais com alimentação, mas destinam a ela uma parcela menor da renda. Vamos examinar a relação nesses dados.

Execute a célula abaixo antes de começar; ela confere a integridade do arquivo e carrega a base.

In [ ]:
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

caminho = "data/ch05/engel_1857.csv"
with open(caminho, "rb") as arquivo:
    print("Arquivo confere com a ficha:", hashlib.sha256(arquivo.read()).hexdigest() == "05b60e091325bc170cb6803ce78f80c029bfe2ae072891ec577f1ebe8deefd61")

engel = pd.read_csv(caminho)

## Exercício 13 — Conhecer a base

Leia a ficha da base. Qual é a unidade de observação? Quais são as variáveis e suas unidades? Use `describe()` para resumir as duas variáveis. Que população esses dados representam — e qual não representam?

## Exercício 14 — Dispersão, covariância e correlação

Faça o gráfico de dispersão da renda e do gasto com alimentação. Calcule a covariância e a correlação. Recalcule a covariância e a correlação com as duas variáveis em milhares de francos e explique as diferenças.

## Exercício 15 — Inclinação

Calcule $b_1 = s_{XY}/s_X^2$ e $b_0 = \bar{y} - b_1\bar{x}$, com a renda como $X$ e o gasto com alimentação como $Y$. Desenhe a reta sobre o gráfico de dispersão e interprete $b_1$ com unidades. Por que essa inclinação não diz quanto uma família específica passaria a gastar se sua renda aumentasse?

## Exercício 16 — A participação da alimentação

Calcule, para cada família, a participação da alimentação na renda (gasto com alimentação dividido pela renda). Resuma essa participação, faça o gráfico de dispersão da renda e da participação e calcule a correlação entre as duas. Os dados são compatíveis com a lei de Engel?

## Exercício 17 — As famílias de renda mais alta

Identifique as famílias com renda acima de 2.500 francos. Recalcule a correlação de Pearson, a de Spearman e a inclinação $b_1$ sem essas famílias. Elas mudam muito os resultados?

## Exercício 18 — Comunicar a lei de Engel

Escreva um texto curto, para um público não técnico, que contenha:

1. a relação entre renda e gasto com alimentação, com a correlação e a inclinação em linguagem acessível;
2. o que acontece com a participação da alimentação no orçamento;
3. os limites da análise — época, população, tipo de dados e leitura descritiva.

### Transferência do Exercício 18 — Qual denominador?

Uma analista comercial descreve a média das participações familiares como a razão entre gasto alimentar total e renda total. Calcule ambas e a diferença em pontos percentuais. Mostre quais pesos de renda reproduzem a razão dos totais. Explique a pergunta de cada medida e por que nenhuma prevê vendas atuais da Beija-flor. Especifique quatro requisitos de dados para segmentação atual, incluindo cobertura da população e a distinção entre gasto familiar e compras na rede.